<table>
  <tr>
       <td width="220" align="center" valign="middle">
      <picture>
        <source media="(prefers-color-scheme: dark)" srcset="https://console.anosys.ai/assets/icon/anosyslogod.svg">
        <source media="(prefers-color-scheme: light)" srcset="https://console.anosys.ai/assets/icon/anosyslogol.svg">
        <img src="https://console.anosys.ai/assets/icon/anosyslogol.svg" width="200" alt="AnoSys Logo">
      </picture>
    </td>
    <td valign="middle">
      <h1 style="margin:0;">AnoSys Core Decorators - Observability PoC</h1>
      <p style="margin:8px 0 0 0;">
        Interactive notebook demonstrating function and pipeline observability with AnoSys decorators.
      </p>
    </td>
  </tr>
</table>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Anosys-AI/anosys-sdk/blob/main/examples/anosys_core_decorators_poc.ipynb)

The **AnoSys Core Decorator** (`@anosys_logger`) provides instant observability for any Python function — whether you are calling custom LLMs (HuggingFace, Ollama, vLLM), executing proprietary tools, or orchestrating data pipelines.

### What you will learn:
1. How to configure `setup_api` and credentials.
2. How to decorate synchronous and asynchronous functions with `@anosys_logger`.
3. How to observe streaming generators (`yield` and `async for yield`).
4. How AnoSys captures exceptions and stack traces (`cvs10`, `cvs11`, `cvs12`) without masking errors.
5. How data privacy levels (`ANOSYS_COLLECTION_LEVEL=metadata`) protect function arguments.



## Step 1: Installation

Install the core AnoSys SDK package from PyPI.


In [ ]:
!pip install -q anosys-sdk-core


## Step 2: Configuration & Credentials

Configure your `ANOSYS_API_KEY` to send traces to your AnoSys dashboard.


In [ ]:
import os

try:
    from google.colab import userdata
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = userdata.get("ANOSYS_API_KEY")
except ImportError:
    import getpass
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = getpass.getpass("AnoSys API Key: ")

from anosys_sdk_core.decorators import setup_api

# Initialize API configuration
setup_api()
print("✅ AnoSys Core Decorator API initialized!")


## Step 3: Decorating Synchronous Functions

Wrap standard Python functions with `@anosys_logger`. AnoSys automatically logs inputs, outputs, caller details, and execution timing in the background without blocking the function.


In [ ]:
from anosys_sdk_core.decorators import anosys_logger

@anosys_logger(source="custom_pipeline.risk_scoring")
def calculate_risk(account_id: str, balance: float, credit_score: int) -> dict:
    """Simulate business logic or custom model inference."""
    risk_level = "LOW" if credit_score > 700 else ("MEDIUM" if credit_score > 600 else "HIGH")
    return {
        "account_id": account_id,
        "risk_level": risk_level,
        "approved_limit": balance * 2.5 if risk_level != "HIGH" else 0.0
    }

result = calculate_risk("ACC-98421", balance=15000.0, credit_score=740)
print("Function Result:", result)
print("✅ Synchronous function trace queued for AnoSys!")


## Step 4: Decorating Asynchronous Functions

`@anosys_logger` transparently supports `async def` coroutines.


In [ ]:
import asyncio

@anosys_logger(source="custom_pipeline.rag_retrieval")
async def fetch_knowledge_documents(query: str, top_k: int = 3) -> list:
    """Simulate async vector retrieval or custom tool execution."""
    await asyncio.sleep(0.05)  # Simulate non-blocking I/O
    return [
        {"id": f"doc-{i}", "title": f"Article {i} for {query}", "relevance": round(0.95 - (i * 0.1), 2)}
        for i in range(1, top_k + 1)
    ]

docs = await fetch_knowledge_documents("hybrid AI privacy architecture", top_k=2)
print("Retrieved Documents:", docs)
print("✅ Asynchronous coroutine trace queued for AnoSys!")


## Step 5: Decorating Streaming Generators

`@anosys_logger` seamlessly captures streaming generator outputs without interrupting item delivery.


In [ ]:
@anosys_logger(source="custom_pipeline.token_stream")
def stream_completion(prompt: str):
    """Simulate local streaming token generation."""
    tokens = ["AnoSys ", "provides ", "unified ", "AI ", "observability."]
    for token in tokens:
        yield token

print("Streaming output: ", end="")
for chunk in stream_completion("Tell me about AnoSys"): 
    print(chunk, end="")
print("\n✅ Generator streaming trace aggregated and queued for AnoSys!")


## Step 6: Automatic Error & Exception Capture

When a decorated function raises an unhandled exception, `@anosys_logger` logs the error type (`cvs10`), error message (`cvs11`), and full stack trace (`cvs12`) to AnoSys, and then safely re-raises the exception for your application's error handling.


In [ ]:
@anosys_logger(source="custom_pipeline.validator")
def validate_transaction(amount: float):
    if amount <= 0:
        raise ValueError(f"Invalid transaction amount: {amount}. Must be positive.")
    return True

try:
    validate_transaction(-250.0)
except ValueError as e:
    print(f"Caught expected exception: {e}")
    print("✅ Error type, message, and stack trace successfully captured in AnoSys telemetry!")


## Step 7: Privacy Controls (`ANOSYS_COLLECTION_LEVEL=metadata`)

Setting `ANOSYS_COLLECTION_LEVEL=metadata` automatically strips function argument values and return payloads, while preserving execution timing, caller info, source labels, and errors.


In [ ]:
from anosys_sdk_core.capture import current_app_level, resolve_app_level

level, source = resolve_app_level()
print(f"Current collection level: {level} (from {source})")
print("When set to 'metadata', function arguments and return objects are dropped before transmission.")
